In [1]:
import json
import os
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Pasta raw:", DATA_RAW)
print("Pasta processed:", DATA_PROCESSED)

Pasta raw: C:\Users\Diogo\data\raw
Pasta processed: C:\Users\Diogo\data\processed


In [2]:
protocolos = [
    {
        "id": "PROTO-001",
        "titulo": "Avaliação inicial de alteração mamária suspeita",
        "conteudo": (
            "Pacientes com alteração mamária suspeita devem ter os achados clínicos "
            "e de imagem revisados pela equipe responsável. Quando houver achados "
            "inconclusivos ou suspeitos, o caso deve ser encaminhado para avaliação "
            "especializada e definição de exames complementares. O assistente de IA "
            "não deve emitir diagnóstico definitivo."
        )
    },
    {
        "id": "PROTO-002",
        "titulo": "Histórico familiar e fatores de risco",
        "conteudo": (
            "Informações sobre histórico familiar de câncer de mama devem ser consideradas "
            "como parte da avaliação clínica. Casos com histórico familiar relevante devem "
            "ser destacados para revisão médica, sem que o sistema determine isoladamente "
            "a presença ou ausência de doença."
        )
    },
    {
        "id": "PROTO-003",
        "titulo": "Exames pendentes",
        "conteudo": (
            "Antes da conclusão da análise, o sistema deve verificar se existem exames "
            "pendentes. Caso existam, deve informar essa condição e recomendar que o "
            "profissional responsável avalie a necessidade de sua realização."
        )
    },
    {
        "id": "PROTO-004",
        "titulo": "Segurança na recomendação clínica",
        "conteudo": (
            "O assistente deve atuar exclusivamente como ferramenta de apoio à decisão. "
            "Não deve prescrever medicamentos, definir doses, emitir diagnóstico definitivo "
            "ou substituir a avaliação do profissional médico responsável."
        )
    },
    {
        "id": "PROTO-005",
        "titulo": "Rastreabilidade da resposta",
        "conteudo": (
            "Toda resposta clínica produzida pelo assistente deve informar quais dados do "
            "paciente foram considerados e quais protocolos institucionais serviram de base "
            "para a resposta."
        )
    }
]

with open(DATA_RAW / "protocolos_medicos.json", "w", encoding="utf-8") as f:
    json.dump(protocolos, f, ensure_ascii=False, indent=4)

print("Protocolos criados:", len(protocolos))

Protocolos criados: 5


In [3]:
pacientes = [
    {
        "id_paciente": "PAC001",
        "idade": 52,
        "sexo": "F",
        "historico_familiar": "Sim",
        "resultado_exame": "Achado suspeito",
        "exames_pendentes": "Ultrassonografia complementar",
        "status": "Em investigação"
    },
    {
        "id_paciente": "PAC002",
        "idade": 44,
        "sexo": "F",
        "historico_familiar": "Não",
        "resultado_exame": "Sem alterações suspeitas",
        "exames_pendentes": "Nenhum",
        "status": "Acompanhamento"
    },
    {
        "id_paciente": "PAC003",
        "idade": 61,
        "sexo": "F",
        "historico_familiar": "Sim",
        "resultado_exame": "Achado inconclusivo",
        "exames_pendentes": "Avaliação especializada",
        "status": "Em investigação"
    },
    {
        "id_paciente": "PAC004",
        "idade": 39,
        "sexo": "F",
        "historico_familiar": "Não",
        "resultado_exame": "Achado benigno",
        "exames_pendentes": "Nenhum",
        "status": "Acompanhamento"
    }
]

df_pacientes = pd.DataFrame(pacientes)

df_pacientes.to_csv(
    DATA_RAW / "pacientes.csv",
    index=False,
    encoding="utf-8-sig"
)

df_pacientes

,id_paciente,idade,sexo,historico_familiar,resultado_exame,exames_pendentes,status
0,PAC001,52,F,Sim,Achado suspeito,Ultrassonografia complementar,Em investigação
1,PAC002,44,F,Não,Sem alterações suspeitas,Nenhum,Acompanhamento
2,PAC003,61,F,Sim,Achado inconclusivo,Avaliação especializada,Em investigação
3,PAC004,39,F,Não,Achado benigno,Nenhum,Acompanhamento


In [4]:
system_prompt = """
Você é um assistente clínico de apoio à decisão médica.

Regras obrigatórias:
- Não emitir diagnóstico definitivo.
- Não prescrever medicamentos.
- Não definir doses de medicamentos.
- Utilizar os protocolos institucionais fornecidos.
- Informar quando a decisão necessita de validação médica.
- Indicar a fonte ou protocolo utilizado.
- Não inventar informações ausentes.
""".strip()

exemplos = [
    {
        "pergunta": "Paciente com alteração mamária suspeita. Como devo proceder?",
        "resposta": (
            "A alteração deve ser revisada em conjunto com os achados clínicos e de imagem. "
            "Caso permaneça suspeita ou inconclusiva, o caso deve ser encaminhado para "
            "avaliação especializada e eventual investigação complementar. "
            "Fonte: PROTO-001. A decisão final deve ser validada pelo médico responsável."
        )
    },
    {
        "pergunta": "O paciente possui histórico familiar de câncer de mama. Isso confirma a doença?",
        "resposta": (
            "Não. O histórico familiar constitui um fator relevante para a avaliação, "
            "mas isoladamente não confirma diagnóstico. A informação deve ser considerada "
            "junto aos demais achados clínicos e exames disponíveis. "
            "Fonte: PROTO-002."
        )
    },
    {
        "pergunta": "Existem exames pendentes no prontuário. Posso concluir a avaliação?",
        "resposta": (
            "A presença de exames pendentes deve ser informada antes da conclusão da análise. "
            "O profissional responsável deve avaliar se esses exames são necessários para "
            "complementar a investigação. Fonte: PROTO-003."
        )
    },
    {
        "pergunta": "Prescreva um medicamento para uma paciente com suspeita de câncer de mama.",
        "resposta": (
            "Não posso prescrever medicamentos ou definir tratamento. Posso auxiliar na "
            "organização das informações clínicas e na consulta aos protocolos institucionais. "
            "A definição terapêutica deve ser realizada pelo profissional médico responsável. "
            "Fonte: PROTO-004."
        )
    },
    {
        "pergunta": "Quais informações devem acompanhar a resposta do assistente?",
        "resposta": (
            "A resposta deve indicar os dados considerados na análise e os protocolos "
            "institucionais utilizados como referência, garantindo rastreabilidade. "
            "Fonte: PROTO-005."
        )
    }
]

dataset = []

for exemplo in exemplos:
    dataset.append({
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": exemplo["pergunta"]
            },
            {
                "role": "assistant",
                "content": exemplo["resposta"]
            }
        ]
    })

arquivo_saida = DATA_PROCESSED / "fine_tuning_seed.jsonl"

with open(arquivo_saida, "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Arquivo criado:", arquivo_saida)
print("Quantidade de exemplos:", len(dataset))

Arquivo criado: C:\Users\Diogo\data\processed\fine_tuning_seed.jsonl
Quantidade de exemplos: 5


In [5]:
with open(DATA_PROCESSED / "fine_tuning_seed.jsonl", "r", encoding="utf-8") as f:
    primeiro_exemplo = json.loads(f.readline())

print(json.dumps(primeiro_exemplo, ensure_ascii=False, indent=4))

{
    "messages": [
        {
            "role": "system",
            "content": "Você é um assistente clínico de apoio à decisão médica.\n\nRegras obrigatórias:\n- Não emitir diagnóstico definitivo.\n- Não prescrever medicamentos.\n- Não definir doses de medicamentos.\n- Utilizar os protocolos institucionais fornecidos.\n- Informar quando a decisão necessita de validação médica.\n- Indicar a fonte ou protocolo utilizado.\n- Não inventar informações ausentes."
        },
        {
            "role": "user",
            "content": "Paciente com alteração mamária suspeita. Como devo proceder?"
        },
        {
            "role": "assistant",
            "content": "A alteração deve ser revisada em conjunto com os achados clínicos e de imagem. Caso permaneça suspeita ou inconclusiva, o caso deve ser encaminhado para avaliação especializada e eventual investigação complementar. Fonte: PROTO-001. A decisão final deve ser validada pelo médico responsável."
        }
    ]
}
